# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saad-aamer/SaadFlyRankInternship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

My rule is to find customers who are 'at risk' of leaving us. I'll define 'at risk' as customers who haven't bought anything in a while but used to spend a lot when they did. This means they are valuable but inactive.

**Reason Codes:**
*   **NO_RECENT_PURCHASE**: The customer hasn't made a purchase in a set number of days.
*   **HIGH_AVG_VALUE**: The customer's average purchase value is above a certain amount.
*   **AT_RISK_COMBO**: Both 'NO_RECENT_PURCHASE' and 'HIGH_AVG_VALUE' apply, marking them as a high-priority at-risk customer.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Define thresholds for the rule
DAYS_SINCE_LAST_PURCHASE_THRESHOLD = 90  # For NO_RECENT_PURCHASE
AVERAGE_PURCHASE_VALUE_THRESHOLD = 50.0  # For HIGH_AVG_VALUE

## 2. Build the ranked queue (writes the CSV)

To build the ranked queue, I'll first create some made-up customer purchase data since we don't have real data. Then, for each customer, I'll figure out two things: how many days it's been since their last purchase and what their average purchase amount is. After that, I'll use the thresholds we set earlier to identify customers who are at risk and assign them a reason code. I'll give them a simple score based on how 'at-risk' they are, sort them by this score, and save the list into a CSV file called `baseline_action_score.csv` in the `work/outputs` folder.

In [ ]:
import pandas as pd
from datetime import datetime, timedelta
import numpy as np
import os

# Reference date for 'today'
TODAY = datetime(2023, 10, 26)

# Generate dummy customer data
np.random.seed(42)
customer_ids = range(1, 101) # 100 customers

data = []
for cust_id in customer_ids:
    num_purchases = np.random.randint(1, 20) # 1 to 20 purchases per customer
    for _ in range(num_purchases):
        purchase_date = TODAY - timedelta(days=np.random.randint(1, 365)) # Purchases within the last year
        purchase_amount = np.random.uniform(5.0, 200.0)
        data.append({'customer_id': cust_id, 'purchase_date': purchase_date, 'purchase_amount': purchase_amount})

customers_df = pd.DataFrame(data)

# Calculate metrics for each customer
customer_summary = customers_df.groupby('customer_id').agg(
    last_purchase_date=('purchase_date', 'max'),
    average_purchase_value=('purchase_amount', 'mean')
).reset_index()

customer_summary['days_since_last_purchase'] = (TODAY - customer_summary['last_purchase_date']).dt.days

# Apply the rule and assign reason codes
def assign_reason_code(row):
    no_recent = row['days_since_last_purchase'] > DAYS_SINCE_LAST_PURCHASE_THRESHOLD
    high_avg = row['average_purchase_value'] > AVERAGE_PURCHASE_VALUE_THRESHOLD

    if no_recent and high_avg:
        return 'AT_RISK_COMBO'
    elif no_recent:
        return 'NO_RECENT_PURCHASE'
    elif high_avg:
        return 'HIGH_AVG_VALUE'
    else:
        return None

customer_summary['reason_code'] = customer_summary.apply(assign_reason_code, axis=1)

# Assign a simple score (higher score means more at-risk)
def assign_score(row):
    if row['reason_code'] == 'AT_RISK_COMBO':
        return 3
    elif row['reason_code'] == 'NO_RECENT_PURCHASE':
        return 2
    elif row['reason_code'] == 'HIGH_AVG_VALUE':
        return 1
    else:
        return 0

customer_summary['risk_score'] = customer_summary.apply(assign_score, axis=1)

# Filter for at-risk customers and rank them
ranked_queue = customer_summary[customer_summary['risk_score'] > 0].sort_values(by='risk_score', ascending=False)

# Create the output directory if it doesn't exist
output_dir = 'work/outputs'
os.makedirs(output_dir, exist_ok=True)

# Save the ranked queue to a CSV file
output_path = os.path.join(output_dir, 'baseline_action_score.csv')
ranked_queue.to_csv(output_path, index=False)

print(f"Ranked queue saved to {output_path}")
print(f"Number of at-risk customers: {len(ranked_queue)}")
print("Top 5 at-risk customers:")
print(ranked_queue.head())

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.